# Single Chunk Debug Notebook

Development sandbox for stepping through a **single chunk** of a textbook chapter.
Uses the `%load` bridge pattern — implementation lives in `src/`, this notebook is for debugging.

**Pipeline:** `check_database` → `draft` → `judge` → `revise` (if needed) → `save_to_db`

**SOLO-74 Updates:** Uses `llama-3.3-70b-versatile` for drafting/revising and `llama-3.1-8b-instant` for judging. Supports round-robin API key rotation via `GROQ_API_KEYS`.

## Setup

In [2]:
%load_ext autoreload
%autoreload 2
import os
import sys
import hashlib
from pathlib import Path
from dotenv import load_dotenv
from rich import print as rprint

load_dotenv()
sys.path.insert(0, str(Path().resolve().parent / 'src'))

from note_taker.database import DatabaseManager
from note_taker.processing import parse_markdown_chunks
from note_taker.pipeline.nodes import check_database_node, outline_draft_node, qa_draft_node, judge_node, revise_node, save_to_db_node
from note_taker.pipeline.state import GraphState
from note_taker.models import FinalArtifactV1

DatabaseManager._instance = None
db = DatabaseManager('dev_notes.db')
db.ensure_database()
rprint('[green]Setup complete[/green]')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Setup complete

## Load Chapter File

In [7]:
DATA_DIR = Path().resolve().parent / 'data' / 'building-applications-with' / 'chapters'
target_file = DATA_DIR / 'chapter_008.md'

rprint(f"Target file exists: [bold]{'✓' if target_file.exists() else '✗'}[/bold]")
rprint(f"File: [cyan]{target_file.name}[/cyan]")

Target file exists: ✓

File: chapter_008.md

## Chunk the Chapter

In [8]:
chunks = parse_markdown_chunks(str(target_file))
rprint(f"Total chunks found: [bold green]{len(chunks)}[/bold green]")

for i, c in enumerate(chunks):
    rprint(f"  [{i:03d}] {c['title']}")

Total chunks found: 23

[000] Chapter 8. From One Agent to Many

[001] How Many Agents Do I Need?

[002] Single-Agent Scenarios - Part 1

[003] Single-Agent Scenarios - Part 2

...

[018] Message Brokers and Event Buses

[019] Actor Frameworks: Ray, Orleans, and Akka

[020] Orchestration and Workflow Engines

[021] Managing State and Persistence

[022] Conclusion

## Pick a Single Chunk

In [9]:
chunk_index = 0  # Change this to select a different chunk
selected_chunk = chunks[chunk_index]

rprint(f"Selected: [bold]{selected_chunk['title']}[/bold]")
rprint(f"Content length: {len(selected_chunk['content'])} chars")
rprint('---')
rprint(selected_chunk['content'][:300] + '...')

Selected: Chapter 8. From One Agent to Many

Content length: 910 chars

---

# Chapter 8. From One Agent to Many

Most use cases start with one agent, but as the number of tools increases, and
the range of problems you want your agent to solve increases, introducing a
multiagent pattern can improve the overall performance and reliability. Just as
we saw that it’s probably no...

## Node: check_database
Check if this chunk has already been processed. Sets `skip_processing` in `GraphState`.

In [10]:
source_hash = hashlib.sha256(selected_chunk['content'].encode('utf-8')).hexdigest()
state: GraphState = {
    "chunk_id": f"BuildingAIAgents:Chapter5:{chunk_index:03d}",
    "source_content": selected_chunk['content'],
    "source_hash": source_hash,
    "force_refresh": True,  # Set False to test skip logic
    "revision_count": 0,
    "artifact": None,
    "skip_processing": False
}

new_state = check_database_node(state, db_manager=db)
rprint(f"Chunk ID   : {state['chunk_id']}")
rprint(f"Source hash: {source_hash[:16]}...")
rprint(f"Skip processing? [bold]{'Yes ⏭' if new_state['skip_processing'] else 'No 🔄'}[/bold]")

Chunk ID   : BuildingAIAgents:Chapter5:000

Source hash: 5cf95587e4d93ecb...

Skip processing? No 🔄

## Node: draft
Generate the initial outline and Q&A pairs. Calls the Groq API.

In [11]:
rprint('[yellow]▶ Running outline_draft_node...[/yellow]')
outline_result = outline_draft_node(new_state)
new_state['outline'] = outline_result['outline']
rprint(f'[green]✓ Outline draft complete[/green]')

rprint('[yellow]▶ Running qa_draft_node...[/yellow]')
qa_result = qa_draft_node(new_state)
artifact = qa_result['artifact']
new_state['artifact'] = artifact

rprint(f'[green]✓ QA draft complete[/green]')
rprint(f'  Outline items: {len(artifact.outline)}')
rprint(f'  Q&A pairs: {len(artifact.qa_pairs)}')
rprint()
for i, qa in enumerate(artifact.qa_pairs):
    rprint(f'  [{i}] Q: {qa.question}')
    rprint(f'      A: {qa.answer[:120]}...')
    rprint()

▶ Running outline_draft_node...

✓ Outline draft complete

▶ Running qa_draft_node...

Retrying note_taker.llm._invoke_single_outlines in 4 seconds as it raised BadRequestError: Error code: 400 - {'error': {'message': "Generated JSON does not match the expected schema. Please adjust your prompt. See 'failed_generation' for more details. Error: jsonschema: '' does not validate with /type: expected object, but got array", 'type': 'invalid_request_error', 'code': 'json_validate_failed', 'failed_generation': '[\n{"qa_pairs":[{"question":"Why is it recommended to start building with a single agent?","answer":"Starting with one agent keeps the system simple, allowing you to understand core behavior before adding complexity.","source_context":"Most use cases start with one agent."},{"question":"What scalability problems can arise from keeping an agent monolithic?","answer":"A monolithic agent can become hard to maintain, less performant, and less reliable as tool count and problem scope grow, similar to a single‑file codebase or server monolith.","source_context":"Just as we sa

✓ QA draft complete

...

A: Assess whether the new functionality introduces high interdependencies or excessive complexity; if so, 
redesign or keep ...

[10] Q: Why are consistent naming conventions and documentation crucial when managing multiple agents?

A: They make it easier for developers to discover, understand, and maintain agents, reducing confusion and 
errors in large ...

[11] Q: What role do deployment and orchestration practices play in operating a multi‑agent system?

A: Proper deployment and orchestration ensure agents are correctly instantiated, scaled, and coordinated, 
enabling reliable...

## Node: judge
Score each Q&A pair on accuracy, clarity, and recall-worthiness (0.0–1.0).

In [12]:
rprint('[yellow]▶ Running judge_node...[/yellow]')
judge_result = judge_node(new_state)
artifact = judge_result['artifact']
new_state['artifact'] = artifact

failing = [qa for qa in artifact.qa_pairs if qa.judge_score is not None and qa.judge_score < 0.7]
rprint(f'[green]✓ Judge complete[/green]')
rprint(f'  Passing: {len(artifact.qa_pairs) - len(failing)} | Failing: {len(failing)}')
rprint()
for i, qa in enumerate(artifact.qa_pairs):
    icon = '✓' if qa.judge_score and qa.judge_score >= 0.7 else '✗'
    rprint(f'  [{i}] {icon} score={qa.judge_score}  feedback={qa.judge_feedback}')

▶ Running judge_node...

✓ Judge complete

Passing: 12 | Failing: 0

[0] ✓ score=1.0  feedback=

[1] ✓ score=1.0  feedback=

...

[7] ✓ score=0.93  feedback=

[8] ✓ score=0.87  feedback=The question could be more specific about what 'signals' refer to.

[9] ✓ score=0.7  feedback=The question and answer could be more precise. What constitutes 'excessive complexity'?

[10] ✓ score=0.9  feedback=

[11] ✓ score=0.93  feedback=

## Node: revise (if needed)
Rewrite Q&A pairs that scored < 0.7 based on judge feedback.

In [13]:
if failing:
    rprint('[yellow]▶ Running revise_node...[/yellow]')
    revise_result = revise_node(new_state)
    artifact = revise_result['artifact']
    new_state['artifact'] = artifact
    rprint(f'[green]✓ Revision complete (count: {revise_result["revision_count"]})[/green]')
    for i, qa in enumerate(artifact.qa_pairs):
        rprint(f'  [{i}] Q: {qa.question}')
else:
    rprint('[green]All pairs passed — no revision needed.[/green]')

All pairs passed — no revision needed.

## Save to Dev DB
Persist the final `FinalArtifactV1` to the development SQLite database.

In [14]:
save_to_db_node(new_state, db_manager=db)
rprint(f'[green]✓ Artifact saved as {new_state["chunk_id"]}[/green]')

# Verify round-trip
retrieved = db.get_artifact(new_state['chunk_id'])
rprint(f'[green]✓ Retrieved from DB: {len(retrieved.qa_pairs)} Q&A pairs, {len(retrieved.outline)} outline items[/green]')

✓ Artifact saved as BuildingAIAgents:Chapter5:000

✓ Retrieved from DB: 12 Q&A pairs, 18 outline items

In [15]:
artifact

FinalArtifactV1(version=1, source_hash='5cf95587e4d93ecbb0b15c0fa95b2b7bdecaf4607863b354376dbe702709b236', outline=[OutlineItem(title='Transition from One Agent to Many', level=1, items=[]), OutlineItem(title='Why Begin with a Single Agent', level=2, items=[]), OutlineItem(title='Scalability Constraints in Monolithic Agents', level=2, items=[]), OutlineItem(title='Benefits of the Multiagent Pattern', level=1, items=[]), OutlineItem(title='Improved Performance through Parallelism', level=2, items=[]), OutlineItem(title='Enhanced Reliability via Redundancy', level=2, items=[]), OutlineItem(title='Architectural Principles for AI Systems', level=1, items=[]), OutlineItem(title='Modularity and Separation of Concerns', level=2, items=[]), OutlineItem(title='Independent Validation and Testing', level=2, items=[]), OutlineItem(title='Agent Decomposition Strategies', level=1, items=[]), OutlineItem(title='Functional Boundaries for Subagents', level=2, items=[]), OutlineItem(title='Reusable Agen